# Smart Farming — Notebook 3: Advisory model deployment

Runs the Qwen crop-advisory service **alone on port 8003**. It selects a 4-bit Qwen3-4B Transformers path when CUDA is available; otherwise it uses Qwen3 GGUF with `llama.cpp` on CPU (1.7B by default).

The advisory service uses the stable Cloudflare Worker gateway for Hindi/Gujarati translation. This is necessary because the translation service is running in a different Colab runtime.

**Prerequisites:** Run Notebook 1, deploy/configure the Cloudflare Worker with KV binding `UPSTREAMS` and secret `GATEWAY_API_KEY`, and add Colab Secrets `HF_TOKEN`, `CF_API_TOKEN`, `CF_ACCOUNT_ID`, `CF_KV_NAMESPACE_ID`, `CF_WORKER_BASE_URL`, and `GATEWAY_API_KEY`. Run Notebook 2 if you want translation to be available.


## 1. Mount Drive and install advisory dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, time, signal, re, json, requests
from google.colab import userdata

DRIVE_ROOT = "/content/drive/MyDrive/SmartFarming"
ADVISORY_DIR = "/content/drive/MyDrive/AdvisoryServer"
os.makedirs(ADVISORY_DIR, exist_ok=True)
assert os.path.isdir(DRIVE_ROOT), "Run Notebook 1 first to sync model-service files to Drive."

os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)
HF_TOKEN = userdata.get("HF_TOKEN") or ""
os.environ["HF_TOKEN"] = HF_TOKEN

req = os.path.join(DRIVE_ROOT, "requirements.txt")
if os.path.isfile(req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.51,<5", "accelerate", "bitsandbytes", "huggingface_hub",
                "fastapi", "uvicorn[standard]", "requests"], check=True)
cpu_wheel = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python>=0.3.12",
     "--prefer-binary", "--extra-index-url",
     "https://abetlen.github.io/llama-cpp-python/whl/cpu"],
    capture_output=True, text=True
)
if cpu_wheel.returncode != 0:
    print("Prebuilt llama-cpp-python wheel unavailable; trying normal pip installation/build.")
    fallback = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python>=0.3.12"],
        capture_output=True, text=True
    )
    if fallback.returncode != 0:
        print(fallback.stderr[-3000:])
        raise RuntimeError("llama-cpp-python installation failed; CPU advisory mode cannot start.")
print("Advisory dependencies installed.")


## 2. Write advisory server to Drive (gateway-aware translation)

In [ ]:
%%writefile /content/drive/MyDrive/AdvisoryServer/advisory_server.py
import os, json, re, threading, copy
os.environ.setdefault("HF_HOME", "/content/hf_cache")

import torch
import requests
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# GPU present: Qwen3-4B in 4-bit through transformers.
# No GPU: a Qwen3 GGUF file through llama.cpp (choose the file with the two environment variables).
USE_GPU = bool(torch.cuda.is_available())
GPU_MODEL_ID = "Qwen/Qwen3-4B"
CPU_GGUF_REPO = os.environ.get("QWEN_GGUF_REPO", "Qwen/Qwen3-1.7B-GGUF")
CPU_GGUF_FILE = os.environ.get("QWEN_GGUF_FILE", "Qwen3-1.7B-Q8_0.gguf")
MODEL_ID = GPU_MODEL_ID if USE_GPU else f"{CPU_GGUF_REPO}/{CPU_GGUF_FILE}"
BACKEND = "transformers-4bit-gpu" if USE_GPU else "llama.cpp-cpu"
MAX_NEW_TOKENS = int(os.environ.get("ADVISORY_MAX_TOKENS", "300" if USE_GPU else "220"))
KB_PATH = os.environ.get("ADVISORY_KB_PATH", "/content/drive/MyDrive/SmartFarming/advisory_kb.json")
CACHE_FILE = os.environ.get("ADVISORY_CACHE_PATH", "/content/drive/MyDrive/advisory_cache.json")
TRANSLATE_URL = os.environ.get("TRANSLATE_URL", "http://127.0.0.1:8002/translate")
GATEWAY_API_KEY = os.environ.get("GATEWAY_API_KEY", "")

# False = the "treatment" text (products and doses) is copied from the knowledge base, never generated.
# True  = the model writes it, and any dose that is not in the knowledge base is replaced by the KB text.
LLM_WRITES_TREATMENT = False
CONF_GATE = 0.70
FIELDS = ["immediate_action", "treatment", "prevention", "monitoring"]

# Fixed texts: have a native speaker review the Hindi and Gujarati wording before release.
DISCLAIMERS = {
    "en": "DISCLAIMER: Confirm the diagnosis and any chemical use with your local KVK or agriculture officer, and follow the product label.",
    "hi": "अस्वीकरण: कोई भी रसायन या दवा इस्तेमाल करने से पहले अपने स्थानीय कृषि विज्ञान केंद्र (KVK) या कृषि अधिकारी से निदान की पुष्टि करें और उत्पाद के लेबल का पालन करें।",
    "gu": "અસ્વીકરણ: કોઈ પણ રસાયણ કે દવાનો ઉપયોગ કરતા પહેલાં તમારા સ્થાનિક કૃષિ વિજ્ઞાન કેન્દ્ર (KVK) અથવા ખેતીવાડી અધિકારી પાસેથી નિદાનની પુષ્ટિ કરો અને ઉત્પાદનના લેબલનું પાલન કરો.",
}

# ---- knowledge base ----
print(f"[Init] Loading knowledge base from {KB_PATH}...")
ADVISORY_KB = {}
if os.path.exists(KB_PATH):
    try:
        with open(KB_PATH, "r", encoding="utf-8") as f:
            ADVISORY_KB = json.load(f)
        print(f"[Init] Loaded {len(ADVISORY_KB)} crop-disease knowledge entries.")
    except Exception as e:
        print(f"[Init] WARNING: could not load knowledge base: {e}")
else:
    print("[Init] WARNING: knowledge base file not found. Every request will get a 'not found' reply.")

# ---- model ----
lock = threading.Lock()
if USE_GPU:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    print(f"[Init] Loading {GPU_MODEL_ID} in 4-bit NF4 on the GPU...")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(GPU_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(GPU_MODEL_ID, quantization_config=bnb, device_map={"": 0}).eval()
else:
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    print(f"[Init] No GPU. Downloading {CPU_GGUF_REPO}/{CPU_GGUF_FILE} ...")
    gguf_path = hf_hub_download(CPU_GGUF_REPO, CPU_GGUF_FILE)
    print(f"[Init] Loading it with llama.cpp on {os.cpu_count()} CPU threads...")
    llm = Llama(model_path=gguf_path, n_ctx=2048, n_threads=os.cpu_count() or 2, verbose=False)

# ---- cache (English advisories only; translation is done per request) ----
cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            cache = json.load(f)
        print(f"[Cache] Loaded {len(cache)} cached advisories.")
    except Exception as e:
        print(f"[Cache] Could not load cache: {e}")

def save_cache():
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
    except Exception:
        pass

# ---- helpers (pure functions, no model needed) ----
NATIVE_DIGITS = str.maketrans("०१२३४५६७८९૦૧૨૩૪૫૬૭૮૯", "01234567890123456789")
DOSE_RE = re.compile(
    r"\d+(?:\.\d+)?\s*(?:ml|gms?|g|kg|litres?|liters?|l|ppm)\b(?:\s*(?:/|per)\s*[a-z]+)?", re.I)

def norm(s):
    return re.sub(r"\s+", "", str(s).lower())

def kb_blob(kb):
    return norm(json.dumps(kb, ensure_ascii=False))

def dose_tokens(text):
    return [norm(m.group(0)) for m in DOSE_RE.finditer(text or "")]

def doses_ok(text, blob):
    """True when every dose-like quantity in the text also appears in the knowledge entry."""
    return all(t in blob for t in dose_tokens(text))

def as_list(x):
    if x is None or x == "":
        return []
    return x if isinstance(x, list) else [x]

def item_text(x):
    if isinstance(x, dict):
        parts = [f"{k.replace('_', ' ')}: {v}" for k, v in x.items() if k != "source" and v not in (None, "", [])]
        return ", ".join(parts)
    return str(x)

def kb_treatment(kb):
    org = [item_text(o) for o in as_list(kb.get("organic_options"))]
    chem = [item_text(c) for c in as_list(kb.get("chemical_options"))]
    parts = []
    if org:
        parts.append("Organic options: " + "; ".join(org) + ".")
    if chem:
        parts.append("Chemical options (follow the product label): " + "; ".join(chem) + ".")
    return " ".join(parts) or "No specific treatment is listed. Consult your local KVK or agriculture officer."

def fallback_fields(kb):
    cult = [item_text(c) for c in as_list(kb.get("cultural_practices"))]
    prec = [item_text(p) for p in as_list(kb.get("precautions"))]
    return {
        "immediate_action": cult[0] if cult else "Separate the affected plants and consult your local KVK.",
        "treatment": kb_treatment(kb),
        "prevention": " ".join(prec or cult) or "Keep the field clean and avoid overcrowding of plants.",
        "monitoring": "Check the field regularly and note whether the symptoms spread. Contact your local KVK if they do.",
    }

def split_sentences(text):
    return [p for p in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if p]

def digits_of(text):
    return re.findall(r"\d+(?:\.\d+)?", (text or "").translate(NATIVE_DIGITS))

def translate_fields(data, lang):
    """Translate sentence by sentence. A field whose numbers change is kept in English."""
    sents, owner = [], []
    for k in FIELDS:
        for s in split_sentences(data[k]):
            sents.append(s)
            owner.append(k)
    if not sents:
        return {k: data[k] for k in FIELDS}, "ok"
    headers = {"Authorization": f"Bearer {GATEWAY_API_KEY}"} if GATEWAY_API_KEY else {}
    r = requests.post(TRANSLATE_URL, json={"texts": sents, "target": lang}, headers=headers, timeout=180)
    r.raise_for_status()
    out = r.json().get("translations", [])
    if len(out) != len(sents):
        raise ValueError("translation length mismatch")
    per = {k: [] for k in FIELDS}
    for k, dst in zip(owner, out):
        per[k].append(dst)
    result, kept_english = {}, []
    for k in FIELDS:
        joined = " ".join(per[k])
        if digits_of(joined) == digits_of(data[k]):
            result[k] = joined
        else:
            result[k] = data[k]
            kept_english.append(k)
    return result, ("partial" if kept_english else "ok")
# ---- end helpers ----

def _chatml(system_prompt, user_prompt):
    # Same layout as the Qwen3 chat template with thinking switched off (empty think block)
    return (f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
            "<|im_start|>assistant\n<think>\n\n</think>\n\n")

_GRAMMARS = {}
def _json_grammar(keys):
    """llama.cpp grammar that forces a JSON object with exactly these string keys (CPU path only)."""
    if not keys:
        return None
    key = tuple(keys)
    if key not in _GRAMMARS:
        try:
            from llama_cpp import LlamaGrammar
            schema = {"type": "object", "properties": {k: {"type": "string"} for k in keys}, "required": list(keys)}
            _GRAMMARS[key] = LlamaGrammar.from_json_schema(json.dumps(schema))
        except Exception as e:
            print(f"[Grammar] not available ({e}); continuing without it")
            _GRAMMARS[key] = None
    return _GRAMMARS[key]

def generate_llm(system_prompt, user_prompt, max_new_tokens=None, json_keys=None):
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    if USE_GPU:
        msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.2, top_p=0.8, top_k=20)
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    kw = dict(max_tokens=max_new_tokens, temperature=0.2, top_p=0.8, top_k=20, stop=["<|im_end|>"])
    grammar = _json_grammar(json_keys)
    if grammar is not None:
        kw["grammar"] = grammar
    out = llm.create_completion(_chatml(system_prompt, user_prompt), **kw)
    return out["choices"][0]["text"].strip()

try:
    generate_llm("You are an agronomy assistant.", "Say Ready.", max_new_tokens=5)
    extra = f" VRAM allocated: {round(torch.cuda.memory_allocated() / 1e6)} MB" if USE_GPU else ""
    print(f"[Init] {MODEL_ID} warmed up ({BACKEND}).{extra}")
except Exception as e:
    print(f"[Init] Warmup notice: {e}")

def build_advisory(kb, req, crop, disease):
    """Returns English advisory fields plus sources. Model output is checked against the KB."""
    chem = as_list(kb.get("chemical_options"))
    org = as_list(kb.get("organic_options"))
    cult = [item_text(c) for c in as_list(kb.get("cultural_practices"))]
    blob = kb_blob(kb)
    keys = FIELDS if LLM_WRITES_TREATMENT else [k for k in FIELDS if k != "treatment"]
    desc = {
        "immediate_action": "One or two sentences: what to do today, considering the severity and weather",
        "treatment": "Treatment using ONLY the products and doses in the facts",
        "prevention": "Two or three sentences from the cultural practices and precautions",
        "monitoring": "What to check, and how often, over the next one to two weeks",
    }
    system_prompt = (
        "You are an agricultural advisory assistant for Indian farmers. "
        "Use ONLY the verified facts provided. Never name a chemical, product or dose that is not in the facts. "
        "Write short, simple sentences a farmer can follow, and keep every value under 40 words. "
        "Reply with one JSON object and nothing else."
    )
    user_prompt = f"""Verified facts:
- Crop: {crop}
- Diagnosis: {disease}
- Symptoms: {kb.get('symptoms')}
- Cultural practices: {', '.join(cult)}
- Organic options: {json.dumps(org, ensure_ascii=False)}
- Chemical options: {json.dumps(chem, ensure_ascii=False)}
- Precautions: {kb.get('precautions')}

Field situation:
- Severity: {req.severity_bucket} ({req.severity_pct}% of leaf area affected)
- Weather: {req.temperature} C, {req.weather_condition}, {req.humidity}% humidity
- Plot area: {req.plot_acres or 'standard field'} acres

Return ONLY a JSON object with exactly these keys:
{json.dumps({k: desc[k] for k in keys}, indent=2)}
"""
    with lock:
        raw = generate_llm(system_prompt, user_prompt, json_keys=keys)
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        parsed = json.loads(m.group(0)) if m else {}
    except Exception:
        parsed = {}
    if not isinstance(parsed, dict):
        parsed = {}

    fb = fallback_fields(kb)
    data = {}
    for k in FIELDS:
        v = parsed.get(k) if k in keys else None
        if isinstance(v, list):
            v = " ".join(str(x) for x in v)
        if not isinstance(v, str) or not v.strip() or not doses_ok(v, blob):
            v = fb[k]
        data[k] = v.strip()

    srcs = [i.get("source") for i in chem + org if isinstance(i, dict) and i.get("source")]
    srcs += [str(s) for s in as_list(kb.get("sources"))]
    data["sources"] = list(dict.fromkeys(str(s) for s in srcs)) or ["Curated knowledge base (advisory_kb.json)"]
    return data

def package(data, lang, **extra):
    out = {"language": "en"}
    out.update({k: data[k] for k in FIELDS})
    out["sources"] = data.get("sources", [])
    out.update(extra)
    status = "not_requested"
    if lang != "en":
        try:
            translated, status = translate_fields(data, lang)
            out["english"] = {k: data[k] for k in FIELDS}
            out.update(translated)
            out["language"] = lang
        except Exception as e:
            status = "unavailable"
            print(f"[Translate] failed: {e}")
    out["translation_status"] = status
    out["safety_disclaimer"] = DISCLAIMERS.get(out["language"], DISCLAIMERS["en"])
    return out

# ---- API ----
app = FastAPI(title="Smart Farming Crop Advisory Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class AdvisoryRequest(BaseModel):
    crop: str
    disease: str
    confidence: float
    severity_pct: float = 0.0
    severity_bucket: str = "Unknown"
    temperature: str = "N/A"
    humidity: str = "N/A"
    weather_condition: str = "N/A"
    plot_acres: float | None = None
    language: str = "en"   # "en", "hi" or "gu"

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "service": "qwen_advisory",
        "model": MODEL_ID,
        "backend": BACKEND,
        "classes_in_kb": len(ADVISORY_KB),
        "llm_writes_treatment": LLM_WRITES_TREATMENT,
        "vram_mb": round(torch.cuda.memory_allocated() / 1e6) if USE_GPU else None,
    }

@app.post("/advise")
def advise(req: AdvisoryRequest):
    lang = (req.language or "en").lower()
    if lang not in ("en", "hi", "gu"):
        lang = "en"
    crop, disease = req.crop.strip(), req.disease.strip()
    kb_key = f"{crop}|{disease}"
    conf = req.confidence / 100 if req.confidence > 1 else req.confidence   # accept 0-1 or 0-100
    is_healthy = disease.lower() == "healthy"

    # Stage 1: rule gate. Low confidence goes to an expert, not to the model.
    if conf < CONF_GATE and not is_healthy:
        data = {
            "immediate_action": f"The automatic diagnosis for {crop} is not confident enough ({conf * 100:.0f}%). Please show the plant to your local KVK or agriculture officer before spraying anything.",
            "treatment": "No chemical treatment is recommended until an expert confirms the diagnosis.",
            "prevention": "Keep the field well drained and separate suspicious plants.",
            "monitoring": "Watch whether the symptoms spread over the next 48 hours.",
            "sources": ["Local KVK / agriculture extension office"],
        }
        return package(data, lang, escalate_to_expert=True, reason="low_confidence")

    kb = ADVISORY_KB.get(kb_key)
    if not isinstance(kb, dict) or not kb:
        data = {
            "immediate_action": f"No verified guidance is stored for {crop} - {disease}. Please consult your local KVK or agriculture officer.",
            "treatment": "No chemical treatment is recommended without expert advice.",
            "prevention": "Keep the field clean and avoid overcrowding of plants.",
            "monitoring": "Check the plants regularly for spreading symptoms.",
            "sources": ["Local KVK / agriculture extension office"],
        }
        return package(data, lang, escalate_to_expert=True, reason="no_knowledge_entry")

    # Stage 2 and 3: constrained generation, checked against the knowledge entry
    cache_key = f"{kb_key}|{req.severity_bucket}|{req.weather_condition}"
    data = cache.get(cache_key)
    if data is None:
        data = build_advisory(kb, req, crop, disease)
        cache[cache_key] = data
        save_cache()
    return package(copy.deepcopy(data), lang)

## 3. Install Cloudflare Tunnel

In [ ]:
cloudflared_path = "/content/cloudflared"
if not os.path.isfile(cloudflared_path):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cloudflared_path], check=True)
os.chmod(cloudflared_path, 0o755)
subprocess.run([cloudflared_path, "--version"], check=True)


## 4. Choose CPU model and start advisory service

In [ ]:
# CPU default is the faster/smaller 1.7B GGUF. Change to "4B" only if RAM permits.
QWEN_CPU_MODEL = "1.7B"
CPU_MODELS = {
    "1.7B": ("Qwen/Qwen3-1.7B-GGUF", "Qwen3-1.7B-Q8_0.gguf"),
    "4B": ("Qwen/Qwen3-4B-GGUF", "Qwen3-4B-Q4_K_M.gguf"),
}
assert QWEN_CPU_MODEL in CPU_MODELS

GATEWAY_BASE = (userdata.get("CF_WORKER_BASE_URL") or "").rstrip("/")
GATEWAY_KEY = userdata.get("GATEWAY_API_KEY") or ""
assert GATEWAY_BASE, "Add CF_WORKER_BASE_URL to Colab Secrets."
assert GATEWAY_KEY, "Add GATEWAY_API_KEY to Colab Secrets."
TRANSLATE_URL = f"{GATEWAY_BASE}/translation/translate"

env = {
    **os.environ,
    "HF_TOKEN": HF_TOKEN,
    "HF_HOME": "/content/hf_cache",
    "QWEN_GGUF_REPO": CPU_MODELS[QWEN_CPU_MODEL][0],
    "QWEN_GGUF_FILE": CPU_MODELS[QWEN_CPU_MODEL][1],
    "TRANSLATE_URL": TRANSLATE_URL,
    "GATEWAY_API_KEY": GATEWAY_KEY,
}

def tail(path, n=40):
    try:
        return "".join(open(path, errors="replace").readlines()[-n:])
    except Exception:
        return "(log not available)"

def stop_matching(pattern):
    result = subprocess.run(["bash", "-lc", f"pgrep -f {pattern!r} || true"],
                            capture_output=True, text=True).stdout.split()
    for item in result:
        try:
            pid = int(item)
            if pid != os.getpid():
                os.kill(pid, signal.SIGTERM)
        except Exception:
            pass

stop_matching("uvicorn advisory_server:app")
time.sleep(2)
advisory_log = "/content/advisory.log"
_advisory_log_handle = open(advisory_log, "w")
advisory_proc = subprocess.Popen(
    ["uvicorn", "advisory_server:app", "--host", "0.0.0.0", "--port", "8003"],
    cwd=ADVISORY_DIR, env=env, stdout=_advisory_log_handle, stderr=subprocess.STDOUT
)

start_time = time.time()
advisory_ok = False
while time.time() - start_time < 1800:
    if advisory_proc.poll() is not None:
        print("Advisory process exited:", advisory_proc.returncode)
        print(tail(advisory_log))
        break
    try:
        r = requests.get("http://127.0.0.1:8003/health", timeout=3)
        if r.status_code == 200:
            print("Advisory ready:", r.json())
            advisory_ok = True
            break
    except requests.RequestException:
        pass
    time.sleep(5)

if not advisory_ok:
    raise RuntimeError("Advisory service failed to start. Inspect /content/advisory.log.")


## 5. Create advisory Quick Tunnel and update Cloudflare KV

In [ ]:
import requests, time, re, json, os, subprocess
def open_tunnel(port, timeout=90):
    log_path = f"/content/tunnel_{port}.log"
    log_handle = open(log_path, "w")
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=log_handle, stderr=subprocess.STDOUT
    )
    deadline = time.time() + timeout
    while time.time() < deadline:
        content = open(log_path, errors="replace").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            return proc, match.group(0), log_path
        if proc.poll() is not None:
            break
        time.sleep(1)
    raise RuntimeError(f"Could not obtain advisory tunnel URL. See {log_path}")

advisory_tunnel_proc, ADVISORY_URL, advisory_tunnel_log = open_tunnel(8003)
print("ADVISORY_URL =", ADVISORY_URL)
with open("/content/drive/MyDrive/service_urls_advisory.json", "w", encoding="utf-8") as f:
    json.dump({"advisory": ADVISORY_URL}, f, indent=2)

import requests
from google.colab import userdata

def colab_secret(name, required=True):
    value = userdata.get(name)
    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret {name}. Open the key icon in Colab Secrets, add it, "
            "and enable notebook access."
        )
    return value

CF_API_TOKEN = colab_secret("CF_API_TOKEN")
CF_ACCOUNT_ID = colab_secret("CF_ACCOUNT_ID")
CF_KV_NAMESPACE_ID = colab_secret("CF_KV_NAMESPACE_ID")

def update_cloudflare_kv(key, value):
    """Create/overwrite one Worker KV value using the Cloudflare API."""
    endpoint = (
        f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}"
        f"/storage/kv/namespaces/{CF_KV_NAMESPACE_ID}/values/{key}"
    )
    response = requests.put(
        endpoint,
        headers={"Authorization": f"Bearer {CF_API_TOKEN}",
                 "Content-Type": "text/plain"},
        data=value.encode("utf-8"),
        timeout=30
    )
    try:
        payload = response.json()
    except Exception:
        payload = {}
    if not response.ok or payload.get("success") is False:
        raise RuntimeError(
            f"Cloudflare KV update failed for {key}: HTTP {response.status_code}; "
            f"{response.text[:500]}"
        )
    print(f"[Cloudflare KV] Updated {key} -> {value}")

update_cloudflare_kv("ADVISORY_URL", ADVISORY_URL)
print("Stable gateway base URL:", (userdata.get("CF_WORKER_BASE_URL") or "").rstrip("/"))


## 6. Test the advisory route through the stable gateway

In [ ]:
GATEWAY_BASE = (userdata.get("CF_WORKER_BASE_URL") or "").rstrip("/")
GATEWAY_KEY = userdata.get("GATEWAY_API_KEY") or ""
headers = {"Authorization": f"Bearer {GATEWAY_KEY}"}
for path in ["/advisory/health"]:
    try:
        r = requests.get(GATEWAY_BASE + path, headers=headers, timeout=30)
        print(path, "HTTP", r.status_code, r.text[:1000])
    except Exception as exc:
        print(path, "FAILED:", exc)

# Optional local smoke test. This checks the API without generating a full advisory.
try:
    r = requests.get("http://127.0.0.1:8003/health", timeout=10)
    print("Local advisory health:", r.json())
except Exception as exc:
    print("Local health check failed:", exc)


## 7. Diagnostics

In [ ]:
for log_path in ["/content/advisory.log", "/content/tunnel_8003.log"]:
    print("\n====", log_path, "====")
    print(tail(log_path, 50))
print("\nGPU status:")
subprocess.run(["bash", "-lc", "nvidia-smi || echo 'No GPU assigned; CPU GGUF backend is selected.'"])
